In [1]:
# import current working directory

import os

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research\model_experiments


In [2]:
# import libraries

import pandas as pd
import numpy as np

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)

In [3]:
# load transformed scaled data

train_df = pd.read_csv(
    "../../artifacts/data_transformation/train.csv"
)

test_df = pd.read_csv(
    "../../artifacts/data_transformation/test.csv"
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (95512, 917)
Test shape : (23878, 917)


In [4]:
# seperate train and test data

X_train = train_df.drop(
    "is_canceled",
    axis=1
)

y_train = train_df["is_canceled"]

X_test = test_df.drop(
    "is_canceled",
    axis=1
)

y_test = test_df["is_canceled"]

In [5]:
# checking their shapes 

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape :", X_test.shape)
print("y_test shape :", y_test.shape)

X_train shape: (95512, 916)
y_train shape: (95512,)
X_test shape : (23878, 916)
y_test shape : (23878,)


In [6]:
# checking targest distribution

print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(
    y_train.value_counts(normalize=True) * 100
)

Training target distribution:
is_canceled
0    60133
1    35379
Name: count, dtype: int64

Training target percentage:
is_canceled
0    62.958581
1    37.041419
Name: proportion, dtype: float64


# Baseline Model

In [7]:
# Naive Bayes baseline model

naive_bayes_model = GaussianNB()

In [8]:
# train the model

naive_bayes_model.fit(X_train,y_train)

,priors,None
,var_smoothing,1e-09


In [9]:
# prediction

nb_pred = naive_bayes_model.predict(X_test)

nb_prob = naive_bayes_model.predict_proba(X_test)[:, 1]

In [10]:
# evaluation

nb_cm = confusion_matrix(
    y_test,
    nb_pred
)

tn, fp, fn, tp = nb_cm.ravel()


nb_accuracy = accuracy_score(
    y_test,
    nb_pred
)

nb_precision = precision_score(
    y_test,
    nb_pred,
    zero_division=0
)

nb_recall = recall_score(
    y_test,
    nb_pred,
    zero_division=0
)

nb_specificity = tn / (tn + fp)

nb_f1 = f1_score(
    y_test,
    nb_pred,
    zero_division=0
)

nb_roc_auc = roc_auc_score(
    y_test,
    nb_prob
)

nb_pr_auc = average_precision_score(
    y_test,
    nb_prob
)

nb_logloss = log_loss(
    y_test,
    nb_prob
)

In [11]:
# display results

print("=" * 55)
print("NAIVE BAYES - BASELINE EVALUATION")
print("=" * 55)

print(f"Accuracy    : {nb_accuracy:.4f}")
print(f"Precision   : {nb_precision:.4f}")
print(f"Recall      : {nb_recall:.4f}")
print(f"Specificity : {nb_specificity:.4f}")
print(f"F1 Score    : {nb_f1:.4f}")
print(f"ROC-AUC     : {nb_roc_auc:.4f}")
print(f"PR-AUC      : {nb_pr_auc:.4f}")
print(f"Log Loss    : {nb_logloss:.4f}")


NAIVE BAYES - BASELINE EVALUATION
Accuracy    : 0.4786
Precision   : 0.4144
Recall      : 0.9875
Specificity : 0.1791
F1 Score    : 0.5838
ROC-AUC     : 0.5836
PR-AUC      : 0.4141
Log Loss    : 18.7822


In [12]:
# store the baseline model

nb_baseline_results = pd.DataFrame({
    "Model": ["Naive Bayes"],
    "Accuracy": [nb_accuracy],
    "Precision": [nb_precision],
    "Recall": [nb_recall],
    "Specificity": [nb_specificity],
    "F1 Score": [nb_f1],
    "ROC-AUC": [nb_roc_auc],
    "PR-AUC": [nb_pr_auc],
    "Log Loss": [nb_logloss]
})

nb_baseline_results.round(4)

,Model,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,Naive Bayes,0.4786,0.4144,0.9875,0.1791,0.5838,0.5836,0.4141,18.7822


In [13]:
# var_smoothing in gaussian

var_smoothing_values = [
    1e-11,
    1e-9,
    1e-7,
    1e-5,
    1e-3,
    1e-1
]

nb_smoothing_results = []

for value in var_smoothing_values:

    model = GaussianNB(
        var_smoothing=value
    )

    model.fit(
        X_train,
        y_train
    )

    test_pred = model.predict(
        X_test
    )

    test_prob = model.predict_proba(
        X_test
    )[:, 1]

    cm = confusion_matrix(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(
        y_test,
        test_pred
    )

    precision = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    specificity = tn / (tn + fp)

    f1 = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_prob
    )

    pr_auc = average_precision_score(
        y_test,
        test_prob
    )

    logloss = log_loss(
        y_test,
        test_prob
    )

    nb_smoothing_results.append({
        "var_smoothing": value,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Log Loss": logloss
    })


In [14]:
# converting in dataframe

nb_smoothing_results_df = pd.DataFrame(
    nb_smoothing_results
)

nb_smoothing_results_df.round(4)

,var_smoothing,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC,PR-AUC,Log Loss
0,0.000,0.4730,0.4120,0.9890,0.1694,0.5817,0.5795,0.4117,18.9901
1,0.000,0.4786,0.4144,0.9875,0.1791,0.5838,0.5836,0.4141,18.7822
2,0.000,0.4879,0.4185,0.9827,0.1968,0.5871,0.5904,0.4181,18.4352
3,0.000,0.5187,0.4326,0.9614,0.2582,0.5968,0.6120,0.4318,17.3061
4,0.001,0.6460,0.5133,0.8566,0.5221,0.6419,0.7787,0.6209,8.5875
5,0.100,0.6363,0.5053,0.8713,0.4980,0.6396,0.8146,0.7513,1.1793


# Hperparameter tunning

In [15]:
# Gridsearch cv

from sklearn.model_selection import GridSearchCV

In [16]:
# parameter Grid

nb_param_grid = {
    "var_smoothing": [
        1e-3,
        1e-2,
        1e-1,
        1
    ]
}

In [ ]:
# Gridsearchcv

nb_grid_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=nb_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

In [18]:
# model fit

nb_grid_search.fit(X_train,y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,GaussianNB()
,param_grid,"{'var_smoothing': [0.001, 0.01, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,priors,None


In [19]:
# best parameter

print("Best Parameters:")
print(nb_grid_search.best_params_)

print("\nBest CV F1 Score:")
print(nb_grid_search.best_score_)

Best Parameters:
{'var_smoothing': 0.01}

Best CV F1 Score:
0.6649759717870289


In [20]:
# best tuned Naive Bayes model

nb_tuned = nb_grid_search.best_estimator_

In [21]:
# find prediction and probability 

nb_tuned_pred = nb_tuned.predict(X_test)

nb_tuned_prob = nb_tuned.predict_proba(X_test)[:, 1]

In [22]:
# confusion matrix

nb_tuned_cm = confusion_matrix(
    y_test,
    nb_tuned_pred
)

In [23]:
tn, fp, fn, tp = nb_tuned_cm.ravel()

In [24]:
# evaluation 

nb_tuned_accuracy = accuracy_score(
    y_test,
    nb_tuned_pred
)

nb_tuned_precision = precision_score(
    y_test,
    nb_tuned_pred,
    zero_division=0
)

nb_tuned_recall = recall_score(
    y_test,
    nb_tuned_pred,
    zero_division=0
)

nb_tuned_specificity = tn / (tn + fp)

nb_tuned_f1 = f1_score(
    y_test,
    nb_tuned_pred,
    zero_division=0
)

nb_tuned_roc_auc = roc_auc_score(
    y_test,
    nb_tuned_prob
)

nb_tuned_pr_auc = average_precision_score(
    y_test,
    nb_tuned_prob
)

nb_tuned_logloss = log_loss(
    y_test,
    nb_tuned_prob
)

In [25]:
# display results

print("=" * 55)
print("NAIVE BAYES - TUNED MODEL EVALUATION")
print("=" * 55)

print(f"Accuracy    : {nb_tuned_accuracy:.4f}")
print(f"Precision   : {nb_tuned_precision:.4f}")
print(f"Recall      : {nb_tuned_recall:.4f}")
print(f"Specificity : {nb_tuned_specificity:.4f}")
print(f"F1 Score    : {nb_tuned_f1:.4f}")
print(f"ROC-AUC     : {nb_tuned_roc_auc:.4f}")
print(f"PR-AUC      : {nb_tuned_pr_auc:.4f}")
print(f"Log Loss    : {nb_tuned_logloss:.4f}")

NAIVE BAYES - TUNED MODEL EVALUATION
Accuracy    : 0.6931
Precision   : 0.5574
Recall      : 0.8323
Specificity : 0.6112
F1 Score    : 0.6677
ROC-AUC     : 0.8175
PR-AUC      : 0.7554
Log Loss    : 2.6096


In [26]:
# baseline vs tuned model

nb_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1 Score",
        "ROC-AUC",
        "PR-AUC",
        "Log Loss"
    ],

    "Baseline": [
        nb_accuracy,
        nb_precision,
        nb_recall,
        nb_specificity,
        nb_f1,
        nb_roc_auc,
        nb_pr_auc,
        nb_logloss
    ],

    "Tuned": [
        nb_tuned_accuracy,
        nb_tuned_precision,
        nb_tuned_recall,
        nb_tuned_specificity,
        nb_tuned_f1,
        nb_tuned_roc_auc,
        nb_tuned_pr_auc,
        nb_tuned_logloss
    ]
})

nb_comparison.round(4)

,Metric,Baseline,Tuned
0,Accuracy,0.4786,0.6931
1,Precision,0.4144,0.5574
2,Recall,0.9875,0.8323
3,Specificity,0.1791,0.6112
4,F1 Score,0.5838,0.6677
5,ROC-AUC,0.5836,0.8175
6,PR-AUC,0.4141,0.7554
7,Log Loss,18.7822,2.6096


In [27]:
# The baseline was heavily biased toward predicting Cancelled